# EEG Torus Fit And Betti Number Check

This notebook adapts the torus-fit and persistent-homology logic from `monkey_torus_fit copy.ipynb` to one mouse cortical EEG session. The target topology is the torus Betti signature $\beta=(1,2,1)$, but the notebook reports the observed signature rather than forcing the expected answer.

## Environment

Run this notebook with the project conda environment:

```bash
/home/nochen/miniconda3/envs/neuralmanifold/bin/python -m pip install ripser persim
/home/nochen/miniconda3/envs/neuralmanifold/bin/python
```

`ripser` is used for persistent homology. `dreimac` is not required because this notebook estimates Betti numbers directly rather than computing toroidal coordinates. For efficiency, each window first forms a 300-point embedded cloud and then uses ripser's deterministic greedy-permutation landmark approximation (`n_perm=150`) with filtration truncation (`thresh=1.5`).

In [ ]:
from __future__ import annotations

import os
import sys
import time
from collections import Counter
from IPython.display import display
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go  # imported for parity with the monkey notebook and optional inspection
from joblib import Parallel, delayed
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy import signal
from tqdm.auto import tqdm

try:
    from ripser import ripser
except ImportError as exc:
    raise ImportError(
        "Install persistent-homology dependencies first: "
        "/home/nochen/miniconda3/envs/neuralmanifold/bin/python -m pip install ripser persim"
    ) from exc

UNIT_DIR = Path.cwd()
if not (UNIT_DIR / "run_sleep_decoding.py").exists():
    UNIT_DIR = Path("/home/nochen/code/NeuralFieldManifold/notebooks/rudra_novak/novak_neurips_rebuttal")
if str(UNIT_DIR) not in sys.path:
    sys.path.insert(0, str(UNIT_DIR))

from run_sleep_decoding import FS, STATE_COLORS, STATE_NAMES, fit_elliptical_torus_3d, process_eeg_signal

DATA_DIR = UNIT_DIR / "eeg_npy_data"
CACHE_DIR = UNIT_DIR / "cache"
TABLE_DIR = UNIT_DIR / "tables"
PLOT_DIR = UNIT_DIR / "plots" / "topology" / "session_m0900_0960"
for path in [CACHE_DIR, TABLE_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SESSION_ID = "session_m0900_0960"
RECORDING_HOUR = int(SESSION_ID.split("_")[1].removeprefix("m")) // 60 + 1
RECORDING_LABEL = f"Recording Hour {RECORDING_HOUR}"
TARGET_BETTI = (1, 2, 1)
EMBEDDING_DIM = 3
TAU_CANDIDATES = np.array([20, 30, 40, 50, 60, 80], dtype=int)
WINDOW_SECONDS = np.array([2.0, 4.0, 8.0], dtype=float)
THRESHOLDS = np.round(np.arange(0.05, 0.5001, 0.025), 3)
N_SUBSAMPLE = 300
RIPSER_N_PERM = int(os.environ.get("EEG_TORUS_RIPSER_N_PERM", 150))
RIPSER_THRESH = float(os.environ.get("EEG_TORUS_RIPSER_THRESH", 1.5))
RNG_SEED = 42

RUN_MODE = os.environ.get("EEG_TORUS_RUN_MODE", "full").strip().lower()
N_PROBE_WINDOWS = 5 if RUN_MODE == "smoke" else 40
N_MAIN_WINDOWS = 10 if RUN_MODE == "smoke" else 200
N_JOBS = int(os.environ.get("EEG_TORUS_N_JOBS", min(8, os.cpu_count() or 1)))
WRITE_ARTIFACTS = RUN_MODE == "full"

print(f"Notebook directory: {UNIT_DIR}")
print(f"Run mode: {RUN_MODE}; probe windows/config={N_PROBE_WINDOWS}; main windows={N_MAIN_WINDOWS}; n_jobs={N_JOBS}")
print(f"Ripser settings: 300-point clouds, n_perm={RIPSER_N_PERM}, thresh={RIPSER_THRESH}")

In [ ]:
signal_raw = np.load(DATA_DIR / f"{SESSION_ID}_signal.npy")
time_sec = np.load(DATA_DIR / f"{SESSION_ID}_time.npy")
states = np.load(DATA_DIR / f"{SESSION_ID}_state.npy").astype(int)

processed, bandpassed = process_eeg_signal(signal_raw, FS)
expected_samples = 60 * 60 * FS

print(f"Loaded {SESSION_ID}: {signal_raw.shape[0]:,} samples at {FS} Hz")
print(f"Expected one-hour sample count: {expected_samples:,}")
print(f"State labels: {len(states)} two-second windows")
for idx, name in enumerate(STATE_NAMES):
    print(f"  {name:>4}: {int(np.sum(states == idx))} windows")

In [ ]:
def plot_eeg_state_overview(processed_signal: np.ndarray, states: np.ndarray, out_path: Path | None = None):
    ds = 40
    t_min = np.arange(len(processed_signal))[::ds] / FS / 60.0
    y = processed_signal[::ds]

    fig, axes = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True, gridspec_kw={"height_ratios": [2.2, 0.55]})
    axes[0].plot(t_min, y, color="black", lw=0.45, alpha=0.85)
    axes[0].set_ylabel("Processed EEG")
    axes[0].set_title(f"{SESSION_ID}: filtered and envelope-normalized EEG")
    axes[0].spines[["top", "right"]].set_visible(False)

    state_img = states[None, :]
    cmap = ListedColormap(STATE_COLORS)
    axes[1].imshow(state_img, aspect="auto", cmap=cmap, interpolation="nearest", extent=[0, len(states) * 2 / 60, 0, 1], vmin=0, vmax=2)
    axes[1].set_yticks([])
    axes[1].set_xlabel("Time (min)")
    axes[1].set_title("Sleep state labels")
    for state, name, color in zip(range(3), STATE_NAMES, STATE_COLORS):
        axes[1].plot([], [], color=color, lw=6, label=name)
    axes[1].legend(frameon=False, ncol=3, loc="upper right", bbox_to_anchor=(1, 2.2))
    axes[1].spines[["top", "right", "left"]].set_visible(False)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
    return fig

fig = plot_eeg_state_overview(processed, states, PLOT_DIR / "eeg_state_overview.png" if WRITE_ARTIFACTS else None)
plt.show()

In [ ]:
def delay_embed(x: np.ndarray, dim: int = EMBEDDING_DIM, tau: int = 40) -> np.ndarray:
    x = np.asarray(x).ravel()
    n = len(x) - (dim - 1) * tau
    if n <= 0:
        raise ValueError(f"Signal segment too short for dim={dim}, tau={tau}")
    rows = np.arange(n)[:, None]
    cols = (dim - 1 - np.arange(dim)) * tau
    return x[rows + cols]


def normalize_cloud(points: np.ndarray) -> np.ndarray:
    centered = points - points.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(centered, axis=1, keepdims=True)
    return centered / (norms + 1e-12)


def sample_starts(n_samples_total: int, window_sec: float, tau: int, n_windows: int, seed: int) -> np.ndarray:
    window_samples = int(round(window_sec * FS))
    needed = window_samples + (EMBEDDING_DIM - 1) * tau
    max_start = n_samples_total - needed
    if max_start <= 0:
        raise ValueError(f"Window {window_sec}s with tau={tau} does not fit in the signal")
    rng = np.random.default_rng(seed)
    return rng.integers(0, max_start + 1, size=n_windows)


def compute_lifetimes_for_window(signal_x: np.ndarray, start: int, window_sec: float, tau: int, seed: int) -> dict[str, object]:
    window_samples = int(round(window_sec * FS))
    needed = window_samples + (EMBEDDING_DIM - 1) * tau
    seg = signal_x[start : start + needed]
    pts = delay_embed(seg, EMBEDDING_DIM, tau)[:window_samples]
    rng = np.random.default_rng(seed)
    if len(pts) > N_SUBSAMPLE:
        idx = rng.choice(len(pts), size=N_SUBSAMPLE, replace=False)
        idx.sort()
        cloud = pts[idx]
    else:
        cloud = pts
    normed = normalize_cloud(cloud).astype(np.float64, copy=False)
    result = ripser(normed, maxdim=2, coeff=41, n_perm=min(RIPSER_N_PERM, len(normed)), thresh=RIPSER_THRESH)
    lifetimes = []
    for dim in range(3):
        dgm = result["dgms"][dim] if dim < len(result["dgms"]) else np.empty((0, 2))
        finite = np.isfinite(dgm[:, 1]) if len(dgm) else np.array([], dtype=bool)
        lt = np.sort((dgm[finite, 1] - dgm[finite, 0]))[::-1] if len(dgm) else np.array([])
        lifetimes.append(lt.astype(np.float64))
    return {
        "start": int(start),
        "start_sec": float(start / FS),
        "window_sec": float(window_sec),
        "tau": int(tau),
        "lifetimes": lifetimes,
    }


def betti_from_lifetimes(lifetimes: list[np.ndarray], threshold: float) -> tuple[int, int, int]:
    beta0 = 1
    beta1 = int(np.sum(lifetimes[1] > threshold)) if len(lifetimes) > 1 else 0
    beta2 = int(np.sum(lifetimes[2] > threshold)) if len(lifetimes) > 2 else 0
    return beta0, beta1, beta2


def top_lifetimes(lifetimes: list[np.ndarray], dim: int, k: int = 5) -> np.ndarray:
    vals = lifetimes[dim] if dim < len(lifetimes) else np.array([])
    return np.pad(vals[:k], (0, max(0, k - len(vals))), constant_values=0)[:k]


def run_lifetime_jobs(signal_x: np.ndarray, starts: np.ndarray, window_sec: float, tau: int, seed_base: int) -> list[dict[str, object]]:
    seeds = np.random.SeedSequence(seed_base).generate_state(len(starts))
    return Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(compute_lifetimes_for_window)(signal_x, int(start), float(window_sec), int(tau), int(seed))
        for start, seed in zip(starts, seeds)
    )


def threshold_summary(records: list[dict[str, object]], thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for thresh in thresholds:
        bettis = np.asarray([betti_from_lifetimes(rec["lifetimes"], float(thresh)) for rec in records], dtype=int)
        target = np.all(bettis == np.asarray(TARGET_BETTI), axis=1)
        rows.append({
            "threshold": float(thresh),
            "n_windows": int(len(bettis)),
            "target_count": int(target.sum()),
            "target_fraction": float(target.mean()) if len(bettis) else np.nan,
            "beta1_eq2_fraction": float(np.mean(bettis[:, 1] == 2)) if len(bettis) else np.nan,
            "beta2_eq1_fraction": float(np.mean(bettis[:, 2] == 1)) if len(bettis) else np.nan,
        })
    return pd.DataFrame(rows)


def smooth3(values: np.ndarray) -> np.ndarray:
    if len(values) < 3:
        return values.astype(float)
    kernel = np.array([1.0, 1.0, 1.0]) / 3.0
    padded = np.pad(values.astype(float), (1, 1), mode="edge")
    return np.convolve(padded, kernel, mode="valid")


def choose_threshold(summary: pd.DataFrame) -> tuple[float, float, int]:
    smoothed = smooth3(summary["target_fraction"].to_numpy(dtype=float))
    peak = float(np.nanmax(smoothed))
    near = smoothed >= max(0.0, peak - 0.05)
    plateau_width = int(near.sum())
    candidate_idx = np.flatnonzero(smoothed == peak)
    thresholds = summary["threshold"].to_numpy(dtype=float)
    # Prefer the monkey-notebook threshold region when multiple thresholds tie.
    best_idx = candidate_idx[np.argmin(np.abs(thresholds[candidate_idx] - 0.30))]
    return float(thresholds[best_idx]), peak, plateau_width


def select_probe_config(config_rows: list[dict[str, object]]) -> pd.DataFrame:
    df = pd.DataFrame(config_rows)
    df = df.sort_values(
        ["peak_target_fraction", "plateau_width", "tau_distance", "runtime_proxy"],
        ascending=[False, False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    return df


In [ ]:
probe_config_rows = []
probe_records_by_config: dict[tuple[int, float], list[dict[str, object]]] = {}

for tau in TAU_CANDIDATES:
    for window_sec in WINDOW_SECONDS:
        seed = RNG_SEED + int(tau) * 1000 + int(window_sec * 100)
        starts = sample_starts(len(processed), float(window_sec), int(tau), N_PROBE_WINDOWS, seed)
        t0 = time.perf_counter()
        records = run_lifetime_jobs(processed, starts, float(window_sec), int(tau), seed)
        elapsed = time.perf_counter() - t0
        summary_df = threshold_summary(records, THRESHOLDS)
        selected_threshold, peak_fraction, plateau_width = choose_threshold(summary_df)
        probe_records_by_config[(int(tau), float(window_sec))] = records
        probe_config_rows.append({
            "tau": int(tau),
            "window_sec": float(window_sec),
            "selected_threshold": selected_threshold,
            "peak_target_fraction": peak_fraction,
            "plateau_width": int(plateau_width),
            "tau_distance": abs(int(tau) - 40),
            "elapsed_sec": float(elapsed),
            "runtime_proxy": float(window_sec),
        })
        print(f"tau={tau:>2}, window={window_sec:g}s -> peak target={peak_fraction:.3f}, threshold={selected_threshold:.3f}, elapsed={elapsed:.1f}s")

probe_config_df = select_probe_config(probe_config_rows)
selected = probe_config_df.iloc[0].to_dict()
print("\nSelected configuration")
display(probe_config_df)
print(selected)

In [ ]:
selected_tau = int(selected["tau"])
selected_window_sec = float(selected["window_sec"])
selected_threshold = float(selected["selected_threshold"])

main_seed = RNG_SEED + 99_000 + selected_tau * 1000 + int(selected_window_sec * 100)
main_starts = sample_starts(len(processed), selected_window_sec, selected_tau, N_MAIN_WINDOWS, main_seed)
t0 = time.perf_counter()
main_records = run_lifetime_jobs(processed, main_starts, selected_window_sec, selected_tau, main_seed)
main_elapsed = time.perf_counter() - t0
main_threshold_df = threshold_summary(main_records, THRESHOLDS)

betti_sweep = np.asarray([betti_from_lifetimes(rec["lifetimes"], selected_threshold) for rec in main_records], dtype=int)
target_mask = np.all(betti_sweep == np.asarray(TARGET_BETTI), axis=1)
mode_signature_raw, mode_count = Counter(map(tuple, betti_sweep)).most_common(1)[0]
mode_signature = tuple(int(x) for x in mode_signature_raw)

torus_fraction = float(target_mask.mean())
print(f"Main run elapsed: {main_elapsed:.1f}s")
print(f"Betti sweep shape: {betti_sweep.shape}")
print(f"Selected threshold: {selected_threshold:.3f}")
print(f"Target β={TARGET_BETTI}: {target_mask.sum()}/{len(betti_sweep)} windows ({torus_fraction:.1%})")
print(f"Dominant observed β={mode_signature}: {mode_count}/{len(betti_sweep)} windows ({mode_count / len(betti_sweep):.1%})")

top_h1 = np.asarray([top_lifetimes(rec["lifetimes"], 1, 5) for rec in main_records])
top_h2 = np.asarray([top_lifetimes(rec["lifetimes"], 2, 5) for rec in main_records])

summary_row = {
    "session_id": SESSION_ID,
    "tau": selected_tau,
    "window_sec": selected_window_sec,
    "threshold": selected_threshold,
    "n_windows": int(len(betti_sweep)),
    "target_beta0": TARGET_BETTI[0],
    "target_beta1": TARGET_BETTI[1],
    "target_beta2": TARGET_BETTI[2],
    "target_count": int(target_mask.sum()),
    "target_fraction": torus_fraction,
    "dominant_beta0": int(mode_signature[0]),
    "dominant_beta1": int(mode_signature[1]),
    "dominant_beta2": int(mode_signature[2]),
    "dominant_count": int(mode_count),
    "dominant_fraction": float(mode_count / len(betti_sweep)),
    "main_elapsed_sec": float(main_elapsed),
}
summary_df = pd.DataFrame([summary_row])
display(summary_df)

In [ ]:
def plot_threshold_sweep(df: pd.DataFrame, selected_threshold: float, out_path: Path | None = None):
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(df["threshold"], df["target_fraction"], "o-", color="#8B0000", lw=2, label="β=(1,2,1)")
    ax.plot(df["threshold"], df["beta1_eq2_fraction"], "s--", color="#D85A30", lw=1.5, label="β1=2")
    ax.plot(df["threshold"], df["beta2_eq1_fraction"], "^--", color="#3266ad", lw=1.5, label="β2=1")
    ax.axvline(selected_threshold, color="black", ls=":", lw=1.2, label=f"selected={selected_threshold:.3f}")
    ax.set_xlabel("Persistence threshold")
    ax.set_ylabel("Fraction of windows")
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(f"Torus Betti Signature Detection ({RECORDING_LABEL})")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=240, bbox_inches="tight")
    return fig

fig = plot_threshold_sweep(main_threshold_df, selected_threshold, PLOT_DIR / "threshold_sweep.png" if WRITE_ARTIFACTS else None)
plt.show()

In [ ]:
def plot_signature_counts(betti: np.ndarray, out_path: Path | None = None):
    counts = Counter(map(tuple, betti))
    items = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    labels = [f"({b0},{b1},{b2})" for (b0, b1, b2), _ in items]
    values = [count for _, count in items]
    colors = ["#8B0000" if sig == TARGET_BETTI else "#B4B2A9" for sig, _ in items]

    fig, ax = plt.subplots(figsize=(max(6.0, 0.65 * len(items)), 4.0))
    bars = ax.bar(np.arange(len(items)), values, color=colors, edgecolor="white", linewidth=0.7)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.8, str(value), ha="center", va="bottom", fontsize=9)
    ax.set_xticks(np.arange(len(items)), labels, rotation=35, ha="right")
    ax.set_ylabel("Windows")
    ax.set_xlabel("Betti signature β=(β0,β1,β2)")
    ax.set_title(f"Betti Signatures at Threshold {selected_threshold:.3f}")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=240, bbox_inches="tight")
    return fig

fig = plot_signature_counts(betti_sweep, PLOT_DIR / "betti_signature_counts.png" if WRITE_ARTIFACTS else None)
plt.show()

In [ ]:
def tau_lifetime_probe_table(records_by_config: dict[tuple[int, float], list[dict[str, object]]], window_sec: float) -> pd.DataFrame:
    rows = []
    for tau in TAU_CANDIDATES:
        records = records_by_config.get((int(tau), float(window_sec)))
        if not records:
            continue
        h1 = np.asarray([top_lifetimes(rec["lifetimes"], 1, 5) for rec in records])
        h2 = np.asarray([top_lifetimes(rec["lifetimes"], 2, 5) for rec in records])
        row = {"tau": int(tau), "window_sec": float(window_sec)}
        for k in range(3):
            row[f"h1_top{k+1}"] = float(h1[:, k].mean())
            row[f"h2_top{k+1}"] = float(h2[:, k].mean())
        rows.append(row)
    return pd.DataFrame(rows)

lifetime_tau_df = tau_lifetime_probe_table(probe_records_by_config, selected_window_sec)

def plot_tau_lifetimes(df: pd.DataFrame, out_path: Path | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
    for k, color, ls in [(1, "black", "-"), (2, "#8B0000", "-"), (3, "#777777", "--")]:
        axes[0].plot(df["tau"], df[f"h1_top{k}"], marker="o", color=color, ls=ls, label=f"{k}")
        axes[1].plot(df["tau"], df[f"h2_top{k}"], marker="o", color=color, ls=ls, label=f"{k}")
    axes[0].set_title("H1 top lifetimes")
    axes[1].set_title("H2 top lifetimes")
    for ax in axes:
        ax.set_xlabel("τ (samples)")
        ax.set_ylabel("Mean lifetime")
        ax.grid(True, alpha=0.2)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(title="rank", frameon=False)
    fig.suptitle(f"Persistence Lifetimes Across τ ({selected_window_sec:g}s windows)", y=1.03)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=240, bbox_inches="tight")
    return fig

fig = plot_tau_lifetimes(lifetime_tau_df, PLOT_DIR / "tau_lifetime_curves.png" if WRITE_ARTIFACTS else None)
plt.show()
display(lifetime_tau_df)

In [ ]:
def choose_example_record(records: list[dict[str, object]], betti: np.ndarray) -> dict[str, object]:
    target_indices = np.flatnonzero(np.all(betti == np.asarray(TARGET_BETTI), axis=1))
    if len(target_indices):
        return records[int(target_indices[0])]
    # Fallback: pick the window with the largest combined support for two H1 loops and one H2 void.
    scores = []
    for rec in records:
        h1 = top_lifetimes(rec["lifetimes"], 1, 2).sum()
        h2 = top_lifetimes(rec["lifetimes"], 2, 1).sum()
        scores.append(h1 + h2)
    return records[int(np.argmax(scores))]


def torus_mesh(entry: dict[str, object], n_surf: int = 48):
    center = np.asarray(entry["center"])
    axis = np.asarray(entry["direction"])
    u_ax = np.asarray(entry["u_axis"])
    v_ax = np.asarray(entry["v_axis"])
    R1 = float(entry["R1"])
    R2 = float(entry["R2"])
    r = float(entry["minor_radius"])
    phi = np.linspace(0, 2 * np.pi, n_surf)
    theta = np.linspace(0, 2 * np.pi, n_surf)
    PHI, THETA = np.meshgrid(phi, theta)
    bx = R1 * np.cos(PHI)
    by = R2 * np.sin(PHI)
    denom = np.sqrt((R2 * np.cos(PHI)) ** 2 + (R1 * np.sin(PHI)) ** 2) + 1e-12
    out_u = R2 * np.cos(PHI) / denom
    out_v = R1 * np.sin(PHI) / denom
    X = center[0] + bx * u_ax[0] + by * v_ax[0] + r * np.cos(THETA) * (out_u * u_ax[0] + out_v * v_ax[0]) + r * np.sin(THETA) * axis[0]
    Y = center[1] + bx * u_ax[1] + by * v_ax[1] + r * np.cos(THETA) * (out_u * u_ax[1] + out_v * v_ax[1]) + r * np.sin(THETA) * axis[1]
    Z = center[2] + bx * u_ax[2] + by * v_ax[2] + r * np.cos(THETA) * (out_u * u_ax[2] + out_v * v_ax[2]) + r * np.sin(THETA) * axis[2]
    return X, Y, Z


def plot_example_torus(record: dict[str, object], out_path: Path | None = None):
    start = int(record["start"])
    window_samples = int(round(record["window_sec"] * FS))
    needed = window_samples + (EMBEDDING_DIM - 1) * int(record["tau"])
    pts = delay_embed(processed[start:start + needed], EMBEDDING_DIM, int(record["tau"]))[:window_samples]
    if len(pts) > 800:
        idx = np.linspace(0, len(pts) - 1, 800, dtype=int)
        fit_pts = pts[idx]
    else:
        fit_pts = pts
    fit = fit_elliptical_torus_3d(fit_pts)
    entry = {**record, "points": fit_pts, **fit}
    X, Y, Z = torus_mesh(entry)

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(fit_pts[:, 0], fit_pts[:, 1], fit_pts[:, 2], color="black", lw=0.8, alpha=0.75)
    ax.plot_surface(X, Y, Z, color="steelblue", alpha=0.18, linewidth=0, shade=True)
    ax.set_xlabel("x(t)")
    ax.set_ylabel("x(t-τ)")
    ax.set_zlabel("x(t-2τ)")
    ax.set_title(
        f"Example EEG Delay Embedding And Elliptical Torus Fit\n"
        f"t={record['start_sec']:.1f}s, τ={record['tau']}, window={record['window_sec']:g}s, "
        f"MSE={fit['mse']:.3g}, inside={fit['frac_inside']:.2f}"
    )
    ax.view_init(elev=24, azim=35)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=240, bbox_inches="tight")
    return fig, entry

example_record = choose_example_record(main_records, betti_sweep)
fig, example_fit = plot_example_torus(example_record, PLOT_DIR / "example_torus_fit.png" if WRITE_ARTIFACTS else None)
plt.show()
print({k: example_fit[k] for k in ["R1", "R2", "minor_radius", "mse", "mean_error", "frac_inside"]})

In [ ]:
if WRITE_ARTIFACTS:
    cache_path = CACHE_DIR / f"eeg_torus_fit_{SESSION_ID}.npz"
    np.savez_compressed(
        cache_path,
        session_id=np.asarray(SESSION_ID),
        betti_sweep=betti_sweep,
        starts=np.asarray([rec["start"] for rec in main_records], dtype=int),
        start_sec=np.asarray([rec["start_sec"] for rec in main_records], dtype=float),
        tau=np.asarray(selected_tau),
        window_sec=np.asarray(selected_window_sec),
        selected_threshold=np.asarray(selected_threshold),
        thresholds=THRESHOLDS,
        n_subsample=np.asarray(N_SUBSAMPLE),
        ripser_n_perm=np.asarray(RIPSER_N_PERM),
        ripser_thresh=np.asarray(RIPSER_THRESH),
        main_threshold_summary=main_threshold_df.to_records(index=False),
        probe_config_summary=probe_config_df.to_records(index=False),
        top_h1=top_h1,
        top_h2=top_h2,
    )
    summary_df.to_csv(TABLE_DIR / "eeg_torus_betti_summary.csv", index=False)
    main_threshold_df.to_csv(TABLE_DIR / "eeg_torus_threshold_sweep.csv", index=False)
    probe_config_df.to_csv(TABLE_DIR / "eeg_torus_probe_config_summary.csv", index=False)
    lifetime_tau_df.to_csv(TABLE_DIR / "eeg_torus_tau_lifetimes.csv", index=False)
    print(f"Wrote {cache_path}")
    print(f"Wrote {TABLE_DIR / 'eeg_torus_betti_summary.csv'}")
    print(f"Wrote plots in {PLOT_DIR}")
else:
    print("Smoke mode: artifacts were not written.")

In [ ]:
# Acceptance checks for the full run.
if WRITE_ARTIFACTS:
    assert signal_raw.shape[0] == 1_440_000, signal_raw.shape
    assert betti_sweep.shape == (200, 3), betti_sweep.shape
    assert (CACHE_DIR / f"eeg_torus_fit_{SESSION_ID}.npz").exists()
    assert (TABLE_DIR / "eeg_torus_betti_summary.csv").exists()
    assert (PLOT_DIR / "threshold_sweep.png").exists()
    assert (PLOT_DIR / "betti_signature_counts.png").exists()
    print("Full-run acceptance checks passed.")
else:
    assert betti_sweep.shape == (10, 3), betti_sweep.shape
    print("Smoke-run acceptance checks passed.")